# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSets, their @id, name & available fields
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets found: {len(record_sets)}\n")
for idx, rs in enumerate(record_sets):
    print(f"Record Set {idx+1} @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    fields = rs.get('fields', [])
    print(f"  Fields (with @id):")
    for field in fields:
        print(f"    - {field['@id']} (name: {field.get('name', '')})")
    print()
# Display @ids for easy reference
all_rs_ids = [rs['@id'] for rs in record_sets]
print(f"RecordSet @ids: {all_rs_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Pick all record sets for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from RecordSet '{rs_id}' with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for '{rs_id}'. Error: {e}")
        dataframes[rs_id] = None

# For demonstration, select the first non-empty DataFrame
example_rs_id = None
for rs_id, df in dataframes.items():
    if df is not None and not df.empty:
        example_rs_id = rs_id
        break

if example_rs_id:
    print(f"\nExample RecordSet for further analysis: {example_rs_id}")
    print(dataframes[example_rs_id].head())
else:
    print("No non-empty record sets found!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Automatically select a numeric field for EDA if possible
df = dataframes[example_rs_id]

# Try to select a numeric field
numeric_field = None
if df is not None:
    for col in df.columns:
        # Check dtype and missing values
        try:
            df_numeric = pd.to_numeric(df[col], errors='coerce')
            num_na = df_numeric.isna().sum()
            # If less than half are NaN, we assume it's a numeric field
            if num_na < len(df_numeric)//2:
                numeric_field = col
                break
        except Exception:
            continue

if numeric_field is None:
    print("No numeric fields found in RecordSet for EDA.")
else:
    print(f"Using numeric field: {numeric_field} (from RecordSet: {example_rs_id})\n")

    # Convert the column to numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].quantile(0.5)  # median threshold as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (median):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by another non-numeric field
    non_numeric = [c for c in df.columns if c != numeric_field and not pd.api.types.is_numeric_dtype(df[c])]
    group_field = non_numeric[0] if non_numeric else None
    if group_field:
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}' in RecordSet: {example_rs_id}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If group_field exists, also plot grouped means
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

Throughout this notebook, we have loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset with `mlcroissant`. We discovered the data structure via RecordSets and fields (all referenced by their `@id` values), extracted records into DataFrames, and performed initial exploratory analysis—including filtering, normalization, grouping, and visualization. This workflow can be easily extended for more advanced analysis or modeling based on specific research questions and the data contained in this Croissant-formatted dataset.